
<div style="text-align: center; line-height: 0; padding-top: 9px;">
  <img
    src="https://databricks.com/wp-content/uploads/2018/03/db-academy-rgb-1200px.png"
    alt="Databricks Learning"
  >
</div>


# 강의 - 평가 메트릭 모델(Judge)의 종류

## 개요

MLflow는 다양한 평가 시나리오와 커스터마이징 수준에 맞춰 설계된 여러 가지 유형의 평가 메트릭 모델(Judge)을 제공합니다. 본 강의에서는 이미 검증을 거친 기본 탑재형 평가부터 완전히 새로 구성하는 맞춤형 평가 로직에 이르기까지, 활용 가능한 평가 모델의 전반을 살펴봅니다.

아울러 일반적인 평가 기준을 위한 기본 탑재형 모델, 비즈니스 규칙을 적용하기 위한 가이드라인 기반 모델, 그리고 특수한 요구사항을 충족하기 위한 맞춤형 모델을 함께 알아봅니다. 종합적인 평가 워크플로우를 구축하기 위해서는 각 유형을 언제, 어떻게 활용해야 하는지 이해하는 것이 매우 중요합니다.

**학습 목표**

본 강의를 마치면 다음을 수행할 수 있습니다.
- 다양한 평가 모델 유형(기본 탑재형, 가이드라인 기반, 맞춤형)의 차이점 이해하기
- 일반적인 평가 기준에 적합한 기본 탑재형 모델 선택하기
- 비즈니스 규칙 적용을 위한 가이드라인 기반 모델의 구현 방법 익히기
- 맞춤형 모델이 필요한 상황을 파악하고 이를 구현하는 방법 알아보기
- 평가 결과에서 피드백(Feedback) 객체와 도출 근거(Rationale)가 갖는 역할 이해하기

## A. 판단 유형 개요

### A1. 평가자(Evaluation Judges)의 종류

| 접근 방식| 커스터마이징 수준| 사용 사례|
|----------------------|------------------------|-----------|
| 내장 평가자(Built-in judges)| 최소| `Correctness`(정확성) 및 `RetrievalGroundedness`검색 기반 타당성)와 같은 내장 스코어러를 사용하여 LLM 평가를 신속하게 시도할 때 활용합니다.|
| 가이드라인 평가자(Guidelines judges)| 중간| 스타일이나 사실성 가이드라인 등, 자연어로 작성된 사용자 지정 규칙을 답변이 통과하거나 실패하는지 점검하는 내장형 판정단입니다.|
| 사용자 지정 평가자(Custom judges)| 최대| 상세한 평가 기준과 피드백 최적화를 적용하여 완전히 맞춤화된 LLM 판정단을 생성합니다. 수치 점수, 카테고리 또는 불리언(True/False) 값을 반환할 수 있습니다.|
| 코드 기반 스코어러(Code-based scorers)| 최대| 정확한 일치 여부(Exact matching), 형식 검증, 성능 지표 등을 평가하는 프로그래밍 방식의 결정론적(Deterministic) 스코어러입니다.|

## B. 공통 기준(Common Criteria)용 기본 제공 평가자(Built-In Judges)

### B1. 연구 기반 검증 평가자(Research-Validated Judges)

MLflow는 일반적인 평가 작업을 위해 연구를 통해 검증된 기본 제공 평가자를 제공합니다. 이 평가자들은 광범위한 연구를 통해 개발되었으며, 인간 전문가의 판단을 기준으로 검증되었고, 특정 평가 기준에 맞게 최적화되었습니다.

### B2. 핵심 기본 제공 평가자(Core Built-in Judges)

**정확성(Correctness)** 

- 제공된 정답(Ground Truth)과 비교하여 모델의 답변이 사실적으로 정확한지 평가합니다.
- 평가 데이터셋 `expectations` 내에 기댓값(예: 예상 답변 또는 예상 사실) 형태의 정답이 있어야 합니다.

**쿼리 연관성(RelevanceToQuery)**

- 답변이 사용자의 쿼리를 직접적이고 적절하게 다루고 있는지 평가합니다.
- 주제를 벗어나거나 겉돌거나 무관한 답변을 식별하는 데 유용합니다. 정답 데이터가 필요하지 않습니다.

**정보 추출 충분성(RetrievalSufficiency)**

- 추출된 컨텍스트(문맥)에 정답을 포함한 올바른 답변을 생성하는 데 필요한 모든 정보가 포함되어 있는지 판단합니다.
- 정답(기댓값)이 필요하며, 생성 품질보다는 정보 추출(Retrieval)의 품질을 평가합니다.

**정보 추출 연관성(RetrievalRelevance)**

- 추출된 문서가 사용자의 쿼리와 연관성이 있는지 평가합니다.
- 정답 데이터가 필요하지 않으며, 최종 답변과 관계없이 오직 정보 추출 단계에만 초점을 맞춥니다.

### B3. 추가 기본 제공 평가자 (Additional Built-in Judges)

**컨텍스트 기반 정합성(RetrievalGroundedness)**

- 모델의 답변이 추출된 컨텍스트에 기반하고 있는지, 근거 없는 사실을 환각(Hallucination)해내지 않는지 확인합니다.
- 정답 데이터가 필요하지 않으며, 답변과 제공된 문서 간의 일치 여부를 평가합니다.

**안전성(Safety)**

- 답변에 유해하거나 불쾌하거나 안전하지 않은 콘텐츠가 없는지 평가합니다.
- 정답 데이터가 필요하지 않으며, 일반적으로 기본적인 콘텐츠 안전성 검사로 사용됩니다.

**가이드라인(Guidelines)**

- 답변이 지정된 자연어 규칙이나 제약 조건(예: 스타일, 어조, 형식 요구사항)을 따르는지 평가합니다.
- 정답 데이터가 필요하지 않습니다.

**기댓값 가이드라인(ExpectationsGuidelines)**

- 답변이 평가 데이터셋에 정의된 예시별 자연어 기준을 충족하는지 평가합니다.
- 사실에 기반한 정답 데이터는 필요하지 않지만, 기댓값에 제공된 예시별 가이드라인에 의존합니다.

### B4. 예시 사용 패턴

```python
from mlflow.genai.scorers import Correctness

correctness_eval = Correctness(
    model="databricks:/foundation-model-endpoint")

correctness_results = mlflow.genai.evaluate(
    data=eval_dataset,
    predict_fn=lambda input: agent.predict({"input": input}),
    scorers=[correctness_eval],
)
```

이 예시는 Databricks 기초 모델 엔드포인트를 사용하는 `Correctness` 채점기를 통해 에이전트를 평가하는 일반적인 패턴을 보여줍니다.

## ⚠️ 데모 체크포인트
<div style="border-left: 4px solid #ff9800; background: #fff3e0; padding: 16px 20px; border-radius: 4px; margin: 16px 0;">
  <div style="display: flex; align-items: flex-start; gap: 12px;">
    <span style="font-size: 24px;"></span>
    <div>
      <strong style="color: #e65100; font-size: 1.1em;">데모 체크포인트</strong>
      <p style="margin: 8px 0 0 0; color: #333;">이 내장 심사위원이 실제로 어떻게 동작하는지 보려면 <strong>2.2 Demo - Using MLflow Built-In Judges</strong>로 이동하세요. 실습을 마치셨다면 다시 이 강의 노트북으로 돌아와 학습을 계속 진행해 주십시오.</p>
    </div>
  </div>
</div>

## C. 가이드라인 평가자 (Guideline Judges)

### C1. 두 가지 유형의 가이드라인 평가자

**1. 글로벌 가이드라인(`Guidelines` 클래스)**

데이터셋의 모든 평가에 동일한 기준을 적용합니다. 글로벌 가이드라인은 어조, 스타일, 형식 요구사항과 같이 모든 테스트 케이스 전반에 걸쳐 일관된 표준을 강제하고자 할 때 이상적입니다.

```python
from mlflow.genai.scorers import Guidelines

tone_guidelines = Guidelines(
    name="professional_tone",
    guidelines=[
        "답변은 전문적이고 비즈니스에 적합한 언어를 사용해야 합니다",
        "답변은 속어, 구어체, 지나치게 무심한 표현을 피해야 한다",
        "응답은 사용자에게 존중하는 태도를 가져야 합니다"
    ],
    model="databricks:/foundation-model-endpoint"
)
```

### C2. 행당 가이드라인

**2. 행별 가이드라인(`ExpectationsGuidelines` 클래스)**

각 예시마다 서로 다른 기준을 적용하며, 시나리오별로 서로 다른 표준이 필요할 때 유용합니다. 데이터셋의 각 행은 `expectations` 필드에 해당 행만의 구체적인 가이드라인을 포함합니다.

```python
from mlflow.genai.scorers import ExpectationsGuidelines

# 데이터셋에는 기대치의 행별 가이드라인이 포함되어 있습니다
# 예시 행:
# {
#   "input": "환불 정책이 어떻게 되나요?"
#   "output": "환불 정책에 따르면..."
#   "expectations": {
#     "guidelines": ["30일 기한을 반드시 언급해야 한다", "영수증 포함 필수"]
#   }
# }

expected_guidelines = ExpectationsGuidelines(
    name="policy_requirements",
    model="databricks:/foundation-model-endpoint"
)
```

이 접근법은 각 입력이 고유한 검증 기준을 요구하는 다양한 사용 사례를 테스트하는 데 이상적입니다.

### C3. 가이드라인 평가자의 장점 및 권장사항

**가이드라인 평가자의 장점:**

- **도메인 전문가 접근성**: 비즈니스 이해관계자가 코딩 없이도 가이드라인을 작성할 수 있습니다.
- **빠른 반복**: 코드 변경 없이 평가 기준을 업데이트할 수 있습니다.
- **해석 가능성**: 자연어로 작성된 가이드라인 자체가 문서 역할을 합니다.
- **유연성**: 코드로 구현하기 까다로운 복잡하고 맥락 의존적인 요구사항을 표현할 수 있습니다.

**가이드라인 작성 팁:**

- 모호한 표현 대신 구체적이고 명확하게 작성하세요. ("답변은 출처 문서를 인용해야 함" vs "답변은 신뢰할 수 있어야 함")
- 객관적으로 검증 가능한 가이드라인을 작성하세요.
- 답변에서 관찰 가능한 속성에 집중하세요.
- 가이드라인이 의도대로 작동하는지 확인하기 위해 여러 예시에서 테스트해 보세요.

## ⚠️ 데모 체크포인트
<div style="border-left: 4px solid #ff9800; background: #fff3e0; padding: 16px 20px; border-radius: 4px; margin: 16px 0;">
  <div style="display: flex; align-items: flex-start; gap: 12px;">
    <span style="font-size: 24px;"></span>
    <div>
      <strong style="color: #e65100; font-size: 1.1em;">데모 체크포인트</strong>
      <p style="margin: 8px 0 0 0; color: #333;"><strong>2.3 Demo - Guideline Judges with MLflow</strong>로 이동해 가이드라인 심사위원을 실습해 보세요. 실습을 마치셨다면 다시 이 강의 노트북으로 돌아와 학습을 계속 진행해 주십시오.</p>
    </div>
  </div>
</div>

## ⚠️ 랩 체크포인트
<div style="border-left: 4px solid #ff9800; background: #fff3e0; padding: 16px 20px; border-radius: 4px; margin: 16px 0;">
  <div style="display: flex; align-items: flex-start; gap: 12px;">
    <span style="font-size: 24px;"></span>
    <div>
      <strong style="color: #e65100; font-size: 1.1em;">랩 체크포인트</strong>
      <p style="margin: 8px 0 0 0; color: #333;"><strong>2.4 Lab - Applying Agent Evaluation</strong>으로 이동해 포괄적인 평가 워크플로우 구현을 연습해 보세요. 끝나면 이 강의 노트북으로 돌아와 학습을 이어가면 됩니다.</p>
    </div>
  </div>
</div>

## D. 사용자 정의 평가자와 코드 기반 스코어러

### D1. 결정론적 평가를 위한 코드 기반 스코어러

기본 제공 평가기가 요구사항에 맞지 않는 경우, MLflow는 코드 기반 스코어러를 통해 사용자 정의 평가 로직을 지원합니다.

```python
from mlflow.genai.scorers import scorer
from mlflow.entities import Feedback

@scorer
def response_length(outputs):
    """응답 길이가 적절한지 확인하세요."""
    word_count = len(str(outputs.get("response", "")).split())

    if 20 <= word_count <= 100:
        return Feedback(
            value="yes",
            rationale=f"응답 길이 ({word_count} words) 적절합니다."
        )
    else:
        return Feedback(
            value="no",
            rationale=f"반응이 너무 {'short' if word_count < 20 else 'long'} ({word_count} words)"
        )
```

사용자 정의 스코어러를 사용하면 기본 제공 평가기가 제공하는 기능 이상의 도메인 특화 검증 로직을 구현할 수 있습니다. `@scorer` 데코레이터는 작성한 함수를 재사용 가능한 평가 컴포넌트로 변환해 줍니다.

### D2. 원시 값을 반환하는 대안적 방법

더 간단한 사용 사례의 경우, 스코어러가 원시(primitive) 값을 직접 반환하도록 할 수 있습니다.

```python
from mlflow.genai.scorers import scorer

@scorer
def response_length(outputs):
    wc = len(str(outputs.get("response", "")).split())
    return "yes" if 20 <= wc <= 100 else "no"
```

이 방식은 더 간결하지만 평가 결과에 대한 근거(rationale)를 제공하지 않으므로, 직관적인 통과/실패(pass/fail) 확인에 더 적합합니다.

`@scorer`는 일반 Python 함수를 MLflow GenAI Scorer로 변환해 줍니다. 이는 `mlflow.genai.evaluate()`가 오프라인에서 실행할 수 있고, 나중에 프로덕션 모니터링용으로 등록할 수 있는 최고 수준의 플러그인 가능 메트릭입니다.

### D3. 사용자 정의 LLM 평가자

정교한 평가를 위한 **사용자 정의 LLM 평가자**:

기본 제공 평가자가 다루지 않는 특수 평가 기준을 위해 자체 LLM 기반 평가자를 구현할 수 있습니다. 여기에는 다음 과정이 포함됩니다.
1. 평가 기준을 명확하게 지정하는 평가 프롬프트 설계
2. 해당 프롬프트를 기반으로 판단을 내리도록 LLM 호출
3. LLM 응답을 구조화된 `Feedback` 객체로 파싱
4. 이 로직을 `mlflow.genai.evaluate()`와 호환되는 함수로 래핑

**사용자 정의 LLM 평가자를 사용해야 하는 경우**

- 애플리케이션만의 고유한 도메인 특화 평가 기준이 필요한 경우
- 기본 제공 평가기가 실제 사용 사례에 중요한 미묘한 차이를 포착하지 못하는 경우
- 자체 표준이나 규정에 따라 평가하는 경우
- 여러 신호나 데이터 소스를 결합하는 평가 로직이 필요한 경우

사용자 정의 평가자는 MLflow의 평가 프레임워크, 트레이싱 및 로깅 인프라와의 통합을 유지하면서도 최고의 유연성을 제공합니다.

## ⚠️ 데모 체크포인트
<div style="border-left: 4px solid #ff9800; background: #fff3e0; padding: 16px 20px; border-radius: 4px; margin: 16px 0;">
  <div style="display: flex; align-items: flex-start; gap: 12px;">
    <span style="font-size: 24px;"></span>
    <div>
      <strong style="color: #e65100; font-size: 1.1em;">데모 체크포인트</strong>
      <p style="margin: 8px 0 0 0; color: #333;"><strong>2.5 Demo - Custom Judges with MLflow</strong>로 이동해 특수 요구에 맞는 맞춤형 심사위원을 만드는 방법을 알아보세요. 실습을 마치면 다시 이 강의 노트북으로 돌아와 학습을 계속 진행해 주십시오.</p>
    </div>
  </div>
</div>

## E. 피드백 객체와 근거

### E1. 구조화된 피드백 객체

```python
Feedback(
    value="yes",        # 이진 합격/불합격 또는 수치 점수
    rationale="답변은 수도를 새크라멘토로 정확히 지칭하고 있습니다...",  # 설명
    metadata={...}      # 추가 구조화 정보
)
```

MLflow 평가자(judge)는 단순한 스칼라 점수가 아닌 구조화된 피드백(Feedback) 객체를 반환합니다. 이러한 구조는 평가 결과를 이해하는 데 필수적인 해석 가능성을 제공합니다.

**핵심 구성 요소:**
- `value`: 실제 점수 또는 판단 결과
- `rationale`: 사람이 읽을 수 있는 설명
- `metadata`: 추가적인 맥락 정보

### E2. 근거(Rationale)가 중요한 이유

- **디버깅**: 단순히 예시가 실패했다는 사실을 넘어, 왜 실패했는지 그 이유를 파악할 수 있습니다.
- **판단의 검증**: 평가자가 올바른 논리로 추론하고 있는지 확인할 수 있습니다.
- **패턴 식별**: 근거에서 공통적인 주제를 찾아내 시스템적인 문제를 파악할 수 있습니다.
- **이해관계자 커뮤니케이션**: 기술 지식이 없는 대상에게도 평가 결과를 명확히 설명할 수 있습니다.

**근거를 효과적으로 활용하는 방법**

1. 실패한 모든 사례의 근거를 검토하여 공통적인 패턴을 파악하세요.
2. 통과된 사례의 근거도 무작위로 점검하여 평가기가 올바르게 추론하고 있는지 확인하세요.
3. 자주 등장하는 근거 문구를 추출하여 실패 유형을 분류하세요.
4. 팀과 평가 결과를 논의할 때 대표적인 근거 예시를 공유하세요.

통과/실패 점수와 상세한 근거의 결합을 통해, MLflow 평가는 정량적(추적 가능한 지표)인 동시에 정성적(이해 가능한 추론)인 분석을 모두 가능하게 합니다.

## F. 핵심 요약

MLflow는 다양한 평가 요구사항을 충족하기 위해 다음과 같은 종합적인 평가자(Judges) 세트를 제공합니다.

1. **내장형 평가자(Built-in judges)**: 정확성, 관련성, 안전성 등 공통 기준에 대해 연구를 통해 검증된 평가 도구
2. **가이드라인 평가자(Guideline judges)**: 전역적 및 개별 사례 단위의 자연어 가이드라인을 활용한 비즈니스 규칙 평가
3. **사용자 정의 코드 기반 스코어러(Custom code-based scorers)**: 특정 요구사항을 위한 확정적(deterministic) 평가 로직
4. **사용자 정의 LLM 평가자(Custom LLM judges)**: 전문 도메인 요구사항에 대한 고도화된 평가
5. **구조화된 피드백 (Structured feedback)**: 평가 근거를 제공하여 해석 가능성 및 디버깅 역량 강화

적절한 평가 메트릭의 조합은 구체적인 평가 요구사항, 확보된 정답 데이터(Ground Truth), 필요한 맞춤화 수준에 따라 달라집니다. 다음 강의에서는 평가 전략과 실제 구현 방법에 대해 자세히 알아보겠습니다.

&copy; 2026 Databricks, Inc. All rights reserved. Apache, Apache Spark, Spark, the Spark Logo, Apache Iceberg, Iceberg, and the Apache Iceberg logo are trademarks of the <a href="https://www.apache.org/" target="_blank">Apache Software Foundation</a>.<br/><br/><a href="https://databricks.com/privacy-policy" target="_blank">Privacy Policy</a> | <a href="https://databricks.com/terms-of-use" target="_blank">Terms of Use</a> | <a href="https://help.databricks.com/" target="_blank">Support</a>